In [40]:
from calendar import day_name

import pandas as pd
import numpy as np
from datetime import timedelta


In [41]:
customers = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_items_dataset.csv')
order_payment = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_payments_dataset.csv')
order_review = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_orders_dataset.csv')
products = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_sellers_dataset.csv')
products_category_name_translation = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/product_category_name_translation.csv')


In [42]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

orders['order_delivered_customer_date'] = pd.to_datetime(
    orders['order_delivered_customer_date']
)

In [43]:
df = order_items.groupby(by='seller_id')
df['order_id'].nunique().sort_values(ascending=False).head(10)


seller_id
6560211a19b47992c3666cc44a7e94c0    1854
4a3ca9315b744ce9f8e9374361493884    1806
cc419e0650a3c5ba77189a1882b7556a    1706
1f50f920176fa81dab994f9023523100    1404
da8622b14eb17ae2831f4ac5b9dab84a    1314
955fee9216a65b617aa5c0531780ce60    1287
7a67c85e85bb2ce8582c35f2203ad736    1160
ea8482cd71df3c1969d7b9473ff13abc    1146
4869f7a5dfa277a7dca6462dcf3b52b2    1132
3d871de0142ce09b7081e2b9d1733cb1    1080
Name: order_id, dtype: int64

In [44]:
df['price'].sum().sort_values(ascending=False).head(10)

seller_id
4869f7a5dfa277a7dca6462dcf3b52b2    229472.63
53243585a1d6dc2643021fd1853d8905    222776.05
4a3ca9315b744ce9f8e9374361493884    200472.92
fa1c13f2614d7b5c4749cbc52fecda94    194042.03
7c67e1448b00f6e969d365cea6b010ab    187923.89
7e93a43ef30c4f03f38b393420bc753a    176431.87
da8622b14eb17ae2831f4ac5b9dab84a    160236.57
7a67c85e85bb2ce8582c35f2203ad736    141745.53
1025f0e2d44d7041d6cf58b6550e0bfa    138968.55
955fee9216a65b617aa5c0531780ce60    135171.70
Name: price, dtype: float64

In [55]:
orders['delivery_time_days'] = (orders['order_delivered_customer_date']- orders['order_purchase_timestamp']).dt.total_seconds() / 86400
seller_delivery = orders[['order_id', 'delivery_time_days']]

In [58]:
seller_delivery = order_items[['order_id', 'seller_id']].merge(seller_delivery,on='order_id',how='left')

In [61]:
seller_avg_delivery = (seller_delivery.groupby('seller_id')['delivery_time_days'].mean().sort_values(ascending=False))
seller_avg_delivery.head()

seller_id
df683dfda87bf71ac3fc63063fba369d    189.863160
8e670472e453ba34a379331513d6aab1     86.001169
586a871d4f1221763fddb6ceefdeb95e     68.621146
4fb41dff7c50136976d1a5cf004a42e2     66.754657
8629a7efec1aab257e58cda559f03ba7     59.226678
Name: delivery_time_days, dtype: float64

In [62]:
# 1. Seller + Order pair ko unique rakho
seller_orders = order_items[
    ['seller_id', 'order_id']
].drop_duplicates()

# 2. Order ki delivery time add karo
seller_delivery = seller_orders.merge(
    orders[['order_id', 'delivery_time_days']],
    on='order_id',
    how='left'
)

# 3. Seller-wise average delivery time
seller_avg_delivery = (
    seller_delivery
    .groupby('seller_id')['delivery_time_days']
    .mean()
    .sort_values(ascending=False)
)

seller_avg_delivery.head(10)

seller_id
df683dfda87bf71ac3fc63063fba369d    189.863160
8e670472e453ba34a379331513d6aab1     86.001169
586a871d4f1221763fddb6ceefdeb95e     68.621146
4fb41dff7c50136976d1a5cf004a42e2     66.754657
8629a7efec1aab257e58cda559f03ba7     59.226678
e09887ca8c7bf8a4621ce481820414ef     57.878368
3da38366e7bd9baf6369071f782ecdf0     53.348426
eebb3372362aa9a46975164bed19a7e7     52.582190
87f3e35268860433e13d577825aada95     51.497106
6f1a1263039c76e68f40a8e536b1da6a     51.000538
Name: delivery_time_days, dtype: float64

In [65]:
seller_stats = (seller_delivery[seller_delivery['seller_id'].str.startswith('df683')]['delivery_time_days'].agg(['count', 'mean', 'median', 'min', 'max']).round(2))

In [67]:
seller_stats

count       1.00
mean      189.86
median    189.86
min       189.86
max       189.86
Name: delivery_time_days, dtype: float64

In [74]:
top_10_slow_sellers = seller_delivery.groupby('seller_id').agg(order_count=('order_id', 'nunique'), avg_delivery_days=('delivery_time_days', 'mean')).query('order_count >= 100').sort_values(by='avg_delivery_days', ascending=False)
top_10_slow_sellers.head(10).reset_index()

,seller_id,order_count,avg_delivery_days
0,7c67e1448b00f6e969d365cea6b010ab,982,22.338338
1,2eb70248d66e0e3ef83659f71b244378,202,18.013339
2,88460e8ebdecbfecb5f9601833981930,248,17.799441
3,06a2c3af7b3aee5d69171b0e14f0ee87,392,17.728176
4,431af27f296bc6519d890aa5a05fdb11,117,17.234933
5,8e6cc767478edae941d9bd9eb778d77a,106,16.615997
6,b33e7c55446eabf8fe1a42d037ac7d6d,152,16.277411
7,cd68562d3f44870c08922d380acae552,125,16.261455
8,855668e0971d4dfd7bef1b6a4133b41b,312,16.261074
9,8160255418d5aaa7dbdc9f4c64ebda44,385,16.249577


In [75]:
temp_df = seller_delivery.groupby('seller_id').agg(order_count=('order_id', 'nunique'), avg_delivery_days=('delivery_time_days', 'mean'))

In [76]:
temp_df.head()

,order_count,avg_delivery_days
seller_id,,
0015a82c2db000af6aaaf3ae2ecb0532,3,10.793885
001cca7ae9ae17fb1caed9dfb1094831,200,13.405805
001e6ad469a905060d959994f1b41e4f,1,NaN
002100f778ceb8431b7a1020ff7ab48f,51,16.386809
003554e2dce176b5555353e4f3555ac8,1,4.646806
